## 6 — MRdeeP (county-level estimates)
Multivariate Multilevel Regression with Deep Generative Post-Stratification, post-stratified at the **county** level.

Pipeline:
1. `insert_data` — encodes survey + county-level benchmark, builds augmented benchmark proportional to ACS population weights
2. `fit` — trains an ensemble of CGANs (Wasserstein loss with gradient penalty)
3. `post_stratify('county_fips')` — generates synthetic micro-data for every demographic × county cell

Output: `outputs/estimates/mrdeep_county_estimates.csv`

In [1]:
# Install runtime deps. Both lines are idempotent — already-installed packages are skipped.
get_ipython().run_line_magic("pip", "install torch --quiet")
get_ipython().run_line_magic("pip", "install \"git+https://github.com/DeepVerseLib/mrdeep.git#subdirectory=python\" --quiet")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

# Force PyTorch backend (mlx not installed in this env)
import os
os.environ["DEEPVERSE_BACKEND"] = "pytorch"

from mrdeep import MRdeeP

DATA_DIR   = Path("../test_data/processed/")
OUTPUT_DIR = Path("../outputs/")
STATE_CSV_NAME = "mrdeep_state_estimates.csv"

STATE_NAMES = {
    "01":"Alabama","02":"Alaska","04":"Arizona","05":"Arkansas","06":"California",
    "08":"Colorado","09":"Connecticut","10":"Delaware","11":"District of Columbia",
    "12":"Florida","13":"Georgia","15":"Hawaii","16":"Idaho","17":"Illinois",
    "18":"Indiana","19":"Iowa","20":"Kansas","21":"Kentucky","22":"Louisiana",
    "23":"Maine","24":"Maryland","25":"Massachusetts","26":"Michigan",
    "27":"Minnesota","28":"Mississippi","29":"Missouri","30":"Montana",
    "31":"Nebraska","32":"Nevada","33":"New Hampshire","34":"New Jersey",
    "35":"New Mexico","36":"New York","37":"North Carolina","38":"North Dakota",
    "39":"Ohio","40":"Oklahoma","41":"Oregon","42":"Pennsylvania",
    "44":"Rhode Island","45":"South Carolina","46":"South Dakota",
    "47":"Tennessee","48":"Texas","49":"Utah","50":"Vermont",
    "51":"Virginia","53":"Washington","54":"West Virginia","55":"Wisconsin",
    "56":"Wyoming",
}

### 1. Load and prepare data

In [3]:
survey_raw = pd.read_csv(DATA_DIR / "climate_survey_responses_recoded.csv",
                          dtype={"state_fips": str, "county_fips": str})
ps_county  = pd.read_csv(DATA_DIR / "poststrat_county.csv",
                          dtype={"state_fips": str, "county_fips": str})

# Demographic variables: county_fips replaces state_fips compared to the state notebook
DEMOG_VARS = ["gender", "race4", "educ_category", "county_fips"]

OUTCOME_COLS = [c for c in survey_raw.columns if c.endswith("_bin")]
print(f"Survey raw:    {survey_raw.shape}")
print(f"Outcome cols:  {len(OUTCOME_COLS)} binary outcomes")
print(f"Poststrat:     {ps_county.shape} | {ps_county['county_fips'].nunique():,} counties")

Survey raw:    (1200, 40)
Outcome cols:  30 binary outcomes
Poststrat:     (99940, 14) | 3,143 counties


In [4]:
# ── Survey: keep only rows with no NaN in any outcome ─────────────────────
survey = (
    survey_raw[DEMOG_VARS + ["state_fips"] + OUTCOME_COLS]
    .dropna(subset=DEMOG_VARS + OUTCOME_COLS)
    .copy()
)
survey["educ_category"] = survey["educ_category"].astype(str)
state_for_survey = survey["state_fips"]
survey = survey.drop(columns=["state_fips"])

# ── Benchmark: county-level cells with population-proportional count column ──
benchmark = ps_county[DEMOG_VARS + ["state_fips", "N_rounded"]].copy()
benchmark["educ_category"] = benchmark["educ_category"].astype(str)
state_for_benchmark = benchmark[["county_fips", "state_fips"]].drop_duplicates()
benchmark = benchmark.drop(columns=["state_fips"])

target_rows = len(benchmark) * 10
benchmark["count"] = np.maximum(
    1,
    (benchmark["N_rounded"] / benchmark["N_rounded"].sum() * target_rows).round()
).astype(int)
benchmark = benchmark.drop(columns=["N_rounded"])

print(f"Survey (complete cases): {len(survey):,}  "
      f"({survey['happening_bin'].mean()*100:.1f}% believe GW is happening)")
print(f"Benchmark: {len(benchmark):,} strata | "
      f"count range [{benchmark['count'].min()}, {benchmark['count'].max()}]")
print(f"Augmented benchmark rows (oversample=1): {benchmark['count'].sum():,}")

Survey (complete cases): 26  (46.2% believe GW is happening)
Benchmark: 99,940 strata | count range [1, 2170]
Augmented benchmark rows (oversample=1): 1,036,124


### 2. Insert data into MRdeeP

In [5]:
mod = MRdeeP(ensembles=3, random_state=42)
mod.insert_data(
    survey    = survey,
    benchmark = benchmark,
    demog_vars = DEMOG_VARS,
    count_col  = "count",
    oversample = 1,
)
print(mod)

MRdeeP (backend=pytorch, ensembles=3)
  Data inserted: True
  Survey: 26 obs, 30 substantive vars, 4 demographic vars
  Augmented benchmark: 1036124 rows
  Fitted: False


### 3. Train CGAN ensemble

In [6]:
mod.fit(
    epochs       = 500,
    patience     = 50,
    batch_size   = 256,
    k            = 32,
    print_runtime = True,
)
print(mod)

Ensemble 1/3
Ensemble 2/3
Ensemble 3/3
Total fit time: 57.4 seconds.
MRdeeP (backend=pytorch, ensembles=3)
  Data inserted: True
  Survey: 26 obs, 30 substantive vars, 4 demographic vars
  Augmented benchmark: 1036124 rows
  Fitted: True
    Noise dim (k): 32
    Ensemble members: 3
    Generated survey: 1036124 rows
    Total fit time: 57.4s


### 4. Post-stratify → county-level estimates for all 30 outcomes

In [7]:
estimates = mod.post_stratify(levels="county_fips")
print(f"Estimates shape: {estimates.shape}  "
      f"({estimates['county_fips'].nunique():,} counties × {len(OUTCOME_COLS)} outcomes)")
estimates[["county_fips", "happening_bin"]].head(10)

Estimates shape: (3143, 31)  (3,143 counties × 30 outcomes)


,county_fips,happening_bin
0,01001,0.448402
1,01003,0.433614
2,01005,0.428543
3,01007,0.442826
4,01009,0.464484
5,01011,0.442320
6,01013,0.437401
7,01015,0.449860
8,01017,0.426304
9,01019,0.443705


In [8]:
# Wide format: one row per county, all 30 outcomes as columns
result = estimates.merge(state_for_benchmark, on="county_fips", how="left")
result["state_name"] = result["state_fips"].map(STATE_NAMES)

# Reorder columns: county_fips, state_fips, state_name, then all *_bin outcomes
front = ["county_fips", "state_fips", "state_name"]
outcome_cols_present = [c for c in OUTCOME_COLS if c in result.columns]
result = result[front + outcome_cols_present]

print(f"Result shape: {result.shape}")
print(result[["county_fips", "state_name", "happening_bin"]].head(10).to_string(index=False))
print(f"\n'happening_bin' — mean: {result['happening_bin'].mean():.4f}  "
      f"min: {result['happening_bin'].min():.4f}  max: {result['happening_bin'].max():.4f}")

Result shape: (3143, 33)
county_fips state_name  happening_bin
      01001    Alabama       0.448402
      01003    Alabama       0.433614
      01005    Alabama       0.428543
      01007    Alabama       0.442826
      01009    Alabama       0.464484
      01011    Alabama       0.442320
      01013    Alabama       0.437401
      01015    Alabama       0.449860
      01017    Alabama       0.426304
      01019    Alabama       0.443705

'happening_bin' — mean: 0.4437  min: 0.4034  max: 0.4838


### 5. Save results

In [9]:
est_dir = OUTPUT_DIR / "estimates"
est_dir.mkdir(parents=True, exist_ok=True)

OUT = est_dir / "mrdeep_county_estimates.csv"
result.to_csv(OUT, index=False)
print(f"Saved → {OUT}  (rows: {len(result):,})")

Saved → ../outputs/estimates/mrdeep_county_estimates.csv  (rows: 3,143)


In [10]:
# ── State-level rollup vs state CSV ───────────────────────────────────────
state_csv = OUTPUT_DIR / "estimates" / STATE_CSV_NAME
if state_csv.exists():
    pop = ps_county.groupby("county_fips")["N_rounded"].sum().reset_index(name="county_pop")
    rollup_input = result[["county_fips", "state_fips", "happening_bin"]].merge(pop, on="county_fips")
    state_rollup = (
        rollup_input.groupby("state_fips")
        .apply(lambda g: np.average(g["happening_bin"], weights=g["county_pop"]),
               include_groups=False)
        .reset_index(name="county_rollup")
    )
    state_rollup["state_name"] = state_rollup["state_fips"].map(STATE_NAMES)
    state_est = pd.read_csv(state_csv, dtype={"state_fips": str})
    if "estimate" not in state_est.columns and "happening_estimate" in state_est.columns:
        state_est = state_est.rename(columns={"happening_estimate": "estimate"})
    cmp = state_rollup.merge(state_est[["state_fips", "estimate"]], on="state_fips", how="left")
    cmp["abs_diff"] = (cmp["county_rollup"] - cmp["estimate"]).abs()
    print(f"State rollup vs {STATE_CSV_NAME}:")
    print(f"  mean |diff|: {cmp['abs_diff'].mean():.4f}")
    print(f"  max  |diff|: {cmp['abs_diff'].max():.4f}")
else:
    print(f"No state CSV at {state_csv} — skipping rollup comparison.")

State rollup vs mrdeep_state_estimates.csv:
  mean |diff|: 0.0117
  max  |diff|: 0.0274
